## Dados simulados: vendas de ingressos na noite de Halloween

A base do RA1 é estática: cada linha é um filme, e nenhum registro novo chega com o tempo. Para aplicar streaming, criamos registros **artificiais** de venda de ingressos em um cinema na noite de 31/10/2022. Os horários e os valores foram inventados, mas os filmes existem na base limpa do RA1 e estavam em cartaz naquele mês.

Cada venda tem três campos:

| Campo | Significado | Exemplo |
|---|---|---|
| `timestamp` | quando a venda aconteceu | 2022-10-31 20:04:00 |
| `filme` | título do filme na base do RA1 | Terrifier 2 |
| `valor` | valor da venda em reais | 30 |

O ingresso custa R$ 30 (inteira) ou R$ 15 (meia), então cada valor corresponde a uma compra de um ou mais ingressos.

In [ ]:
import pandas as pd

URL_LIMPO = "https://raw.githubusercontent.com/carlaferraz/HorrorMoviesData/refs/heads/main/horror_movies_limpo.csv"
filmes_base = pd.read_csv(URL_LIMPO, usecols=["id", "title", "release_date", "vote_average", "vote_count"])

filmes_em_cartaz = ["Smile", "Halloween Ends", "Terrifier 2", "Barbarian"]
ids_em_cartaz = [882598, 616820, 663712, 913290]

filmes_base[filmes_base["id"].isin(ids_em_cartaz)]

**Leitura:** os quatro filmes estão na base limpa e foram lançados entre setembro e outubro de 2022, então faz sentido que estivessem em cartaz no Halloween. Usamos o `id` porque existe mais de um filme chamado *Smile* na base (Q1 mostrou que título não identifica o filme).

In [ ]:
vendas_etapa1 = [
    ("2022-10-31 20:01:00", "Smile", 60),
    ("2022-10-31 20:03:00", "Halloween Ends", 45),
    ("2022-10-31 20:04:00", "Terrifier 2", 30),
]

vendas_etapa2 = [
    ("2022-10-31 20:06:00", "Barbarian", 90),
    ("2022-10-31 20:07:00", "Smile", 15),
]

vendas_etapa3 = [
    ("2022-10-31 20:08:00", "Halloween Ends", 60),
    ("2022-10-31 20:09:00", "Terrifier 2", 45),
    ("2022-10-31 20:10:00", "Smile", 30),
]

todas_vendas = vendas_etapa1 + vendas_etapa2 + vendas_etapa3
assert all(filme in filmes_em_cartaz for _, filme, _ in todas_vendas)

pd.DataFrame(todas_vendas, columns=["timestamp", "filme", "valor"])

**Leitura:** são 8 vendas entre 20:01 e 20:10, somando R$ 375. Elas chegam em três etapas, como se o sistema do cinema enviasse as vendas aos poucos:

| Etapa | Vendas que chegam | Receita da etapa (R$) |
|---|---|---:|
| 1 | 20:01 Smile (60), 20:03 Halloween Ends (45), 20:04 Terrifier 2 (30) | 135 |
| 2 | 20:06 Barbarian (90), 20:07 Smile (15) | 105 |
| 3 | 20:08 Halloween Ends (60), 20:09 Terrifier 2 (45), 20:10 Smile (30) | 135 |

As vendas estão a um ou dois minutos de distância, então várias caem em mais de uma janela, e cada etapa muda janelas que já existiam além de criar novas.

## Janelas temporais

**Tipo:** Sliding Window (janela deslizante).

**Duração:** 5 minutos. **Deslocamento:** 2 minutos. No Spark: `window("timestamp", "5 minutes", "2 minutes")`.

**Agregação:** soma de `valor` em cada janela, que dá a receita do cinema naquele intervalo. O filme não entra no agrupamento: a pergunta é quanto o cinema arrecadou, não quanto cada filme arrecadou.

**Como as janelas são formadas:** uma nova janela começa a cada 2 minutos (20:00, 20:02, 20:04...) e cada uma dura 5 minutos. O início está incluído e o fim está excluído: a janela 20:00–20:05 conta uma venda das 20:04, mas não uma das 20:05. Como o deslocamento é menor que a duração, as janelas se sobrepõem, e uma venda entra em 2 ou 3 janelas. A venda das 20:04, por exemplo, está em 20:00–20:05, 20:02–20:07 e 20:04–20:09.

**Por que essa escolha:**
- Na noite de Halloween as compras se concentram nos minutos antes de cada sessão. Uma janela de 5 minutos pega um desses picos inteiro sem misturar com o próximo.
- O deslocamento de 2 minutos atualiza a receita dos "últimos 5 minutos" com frequência. Com uma Tumbling Window de 5 minutos, o resultado só mudaria de 5 em 5 minutos e um pico que caísse na divisa entre duas janelas ficaria partido ao meio.
- Duração e deslocamento são os mesmos do notebook da aula, o que facilita comparar o resultado.

**Cuidado na leitura:** como a mesma venda aparece em várias janelas, somar as receitas de todas as janelas não dá a receita da noite. A soma das janelas na etapa 3 é R$ 960, mas o cinema vendeu R$ 375.

### Gabarito: resultado esperado em cada etapa

Calculado à mão, antes de rodar o streaming, para conferir a saída do Spark. A tabela é acumulada: cada etapa mostra todas as janelas com as vendas que chegaram até ali. Em negrito, o que mudou em relação à etapa anterior.

**Etapa 1** (4 janelas)

| Janela | Vendas incluídas | Receita (R$) |
|---|---|---:|
| 19:58–20:03 | 20:01 (60) | 60 |
| 20:00–20:05 | 20:01 (60), 20:03 (45), 20:04 (30) | 135 |
| 20:02–20:07 | 20:03 (45), 20:04 (30) | 75 |
| 20:04–20:09 | 20:04 (30) | 30 |

A venda das 20:03 não entra em 19:58–20:03 porque o fim da janela é excluído.

**Etapa 2** (5 janelas)

| Janela | Vendas incluídas | Receita (R$) |
|---|---|---:|
| 19:58–20:03 | 20:01 (60) | 60 |
| 20:00–20:05 | 20:01 (60), 20:03 (45), 20:04 (30) | 135 |
| 20:02–20:07 | 20:03 (45), 20:04 (30), **20:06 (90)** | **165** |
| 20:04–20:09 | 20:04 (30), **20:06 (90), 20:07 (15)** | **135** |
| **20:06–20:11** | **20:06 (90), 20:07 (15)** | **105** |

A venda das 20:06 aumenta duas janelas antigas e abre uma nova.

**Etapa 3** (7 janelas)

| Janela | Vendas incluídas | Receita (R$) |
|---|---|---:|
| 19:58–20:03 | 20:01 (60) | 60 |
| 20:00–20:05 | 20:01 (60), 20:03 (45), 20:04 (30) | 135 |
| 20:02–20:07 | 20:03 (45), 20:04 (30), 20:06 (90) | 165 |
| 20:04–20:09 | 20:04 (30), 20:06 (90), 20:07 (15), **20:08 (60)** | **195** |
| 20:06–20:11 | 20:06 (90), 20:07 (15), **20:08 (60), 20:09 (45), 20:10 (30)** | **240** |
| **20:08–20:13** | **20:08 (60), 20:09 (45), 20:10 (30)** | **135** |
| **20:10–20:15** | **20:10 (30)** | **30** |

A janela 20:06–20:11 é o pico da noite (R$ 240), porque reúne cinco vendas seguidas. As três primeiras janelas não mudam mais, porque nenhuma venda nova cai nelas.